# Create clinically meaningful pair subsets (ISIC2018)

We create:
- Subset 1: GT-based subset (e.g., MEL or NV)
- Subset 2: Confusing subset (model uncertain between MEL and NV)
- Subset 3: Error subset (GT MEL predicted NV or vice versa)

Outputs are CSV files you can use in CAM scripts.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# --- paths
REPO_ROOT = Path("..")

CSV = REPO_ROOT / "data" / "isic2018" / "val_gt.csv"
IMG_DIR = REPO_ROOT / "data" / "isic2018" / "images_val"
CKPT_PATH = REPO_ROOT / "external" / "weights" / "isic7_last_effnetb4.pth"
OUT_PRED = REPO_ROOT / "outputs" / "preds_isic7_val.csv"
CSV_OUT = REPO_ROOT / "data" / "isic2018" / "subsets"
CSV_OUT.mkdir(parents=True, exist_ok=True)
OUT_PRED.parent.mkdir(parents=True, exist_ok=True)

print("CSV:", CSV.exists())
print("IMG_DIR:", IMG_DIR.exists())
print("CKPT_PATH:", CKPT_PATH.exists())
print("Will write:", OUT_PRED)
print("OUT DIR:", CSV_OUT)

CSV: True
IMG_DIR: True
CKPT_PATH: True
Will write: ../outputs/preds_isic7_val.csv
OUT DIR: ../data/isic2018/subsets


In [3]:
# Load GT CSV + convert one-hot to label
df = pd.read_csv(CSV)
print(df.columns.tolist())
df.head()

['image', 'MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']


,image,MEL,NV,BCC,AKIEC,BKL,DF,VASC
0,ISIC_0034321,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,ISIC_0034322,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,ISIC_0034323,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,ISIC_0034324,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,ISIC_0034325,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [4]:
class_cols = ["MEL","NV","BCC","AKIEC","BKL","DF","VASC"]

# Convert one-hot to label name
# idxmax(axis=1) looks across each row and returns the column name with the largest value
df["gt_label"] = df[class_cols].idxmax(axis=1)

# Image id column expected as "image" without .jpg
assert "image" in df.columns, "CSV must have column 'image'"

df[["image","gt_label"]].head()

,image,gt_label
0,ISIC_0034321,NV
1,ISIC_0034322,NV
2,ISIC_0034323,BCC
3,ISIC_0034324,NV
4,ISIC_0034325,NV


## Create Model Predictions CSV

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import torchvision.transforms as T

# --- device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", device)

# --- load model (efficientnet_pytorch)
from efficientnet_pytorch import EfficientNet

model = EfficientNet.from_name("efficientnet-b4")
model._fc = torch.nn.Linear(model._fc.in_features, 7)

state = torch.load(CKPT_PATH, map_location="cpu")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

# remove possible "module." prefix
cleaned = {}
for k, v in state.items():
    if k.startswith("module."):
        k = k[len("module."):]
    cleaned[k] = v

missing, unexpected = model.load_state_dict(cleaned, strict=False)
print("Missing:", len(missing), "Unexpected:", len(unexpected))

model.eval()
model = model.to(device)

# --- preprocessing (match training)
IMAGE_SIZE = 380
preprocess = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

# --- label order used by the checkpoint
idx_to_label = {0:"AK", 1:"BCC", 2:"BKL", 3:"DF", 4:"MEL", 5:"NV", 6:"VASC"}

# --- run inference
df = pd.read_csv(CSV)
rows = []

with torch.no_grad():
    for i, row in df.iterrows():
        image_id = row["image"]
        img_path = IMG_DIR / f"{image_id}.jpg"
        if not img_path.exists():
            continue

        img = Image.open(img_path).convert("RGB")
        x = preprocess(img).unsqueeze(0).to(device)

        logits = model(x) # (1,7)
        probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy() # (7,)

        pred_idx = int(np.argmax(probs))
        pred_label = idx_to_label[pred_idx]

        rows.append({
            "image": image_id,
            "pred_label": pred_label,
            "p_AK": float(probs[0]),
            "p_BCC": float(probs[1]),
            "p_BKL": float(probs[2]),
            "p_DF": float(probs[3]),
            "p_MEL": float(probs[4]),
            "p_NV": float(probs[5]),
            "p_VASC": float(probs[6]),
        })

pred_df = pd.DataFrame(rows)
pred_df.to_csv(OUT_PRED, index=False)
print("Saved:", OUT_PRED, "rows:", len(pred_df))
pred_df.head()

CSV_VAL: True
IMG_DIR: True
CKPT_PATH: True
Will write: ../outputs/preds_isic7_val.csv
Device: mps
Missing: 0 Unexpected: 0
Saved: ../outputs/preds_isic7_val.csv rows: 193


,image,pred_label,p_AK,p_BCC,p_BKL,p_DF,p_MEL,p_NV,p_VASC
0,ISIC_0034321,NV,2.286867e-11,1.011446e-01,1.297704e-12,2.657453e-05,2.220816e-03,8.966077e-01,2.108439e-07
1,ISIC_0034322,NV,2.118816e-17,5.147772e-17,1.295538e-16,2.919001e-11,2.417301e-16,1.000000e+00,3.513314e-14
2,ISIC_0034323,BCC,1.273256e-14,1.000000e+00,3.379333e-14,2.895610e-12,2.905241e-12,9.124191e-16,6.505454e-14
3,ISIC_0034324,NV,3.302397e-15,2.447337e-15,2.801896e-15,6.886724e-12,2.407070e-11,1.000000e+00,1.169020e-12
4,ISIC_0034325,NV,5.150996e-16,1.016232e-15,7.281745e-15,1.389805e-10,3.586748e-12,1.000000e+00,4.977890e-13


In [5]:
PRED_CSV = REPO_ROOT / "outputs" / "preds_isic7_val.csv"
print("Pred CSV exists:", PRED_CSV.exists())

Pred CSV exists: True


In [6]:
pred = pd.read_csv(PRED_CSV)

# expected probability columns
prob_cols = ["p_AK","p_BCC","p_BKL","p_DF","p_MEL","p_NV","p_VASC"]
missing = [c for c in prob_cols if c not in pred.columns]
if missing:
    raise ValueError(f"Missing columns in pred CSV: {missing}")

In [7]:
class_cols = ["MEL","NV","BCC","AKIEC","BKL","DF","VASC"]
df["gt_label"] = df[class_cols].idxmax(axis=1)

In [8]:
prob_cols = ["p_AK","p_BCC","p_BKL","p_DF","p_MEL","p_NV","p_VASC"]

merged = df.merge(pred, on="image", how="inner")
print("Merged rows:", len(merged))

merged[["image","gt_label","pred_label"] + prob_cols].head()

Merged rows: 193


,image,gt_label,pred_label,p_AK,p_BCC,p_BKL,p_DF,p_MEL,p_NV,p_VASC
0,ISIC_0034321,NV,NV,2.286867e-11,1.011446e-01,1.297704e-12,2.657453e-05,2.220816e-03,8.966077e-01,2.108439e-07
1,ISIC_0034322,NV,NV,2.118816e-17,5.147772e-17,1.295538e-16,2.919001e-11,2.417301e-16,1.000000e+00,3.513314e-14
2,ISIC_0034323,BCC,BCC,1.273256e-14,1.000000e+00,3.379333e-14,2.895610e-12,2.905241e-12,9.124191e-16,6.505454e-14
3,ISIC_0034324,NV,NV,3.302397e-15,2.447337e-15,2.801896e-15,6.886724e-12,2.407070e-11,1.000000e+00,1.169020e-12
4,ISIC_0034325,NV,NV,5.150996e-16,1.016232e-15,7.281745e-15,1.389805e-10,3.586748e-12,1.000000e+00,4.977890e-13


## Subset 1: GT MEL or NV

In [9]:
subset1 = merged[merged["gt_label"].isin(["MEL","NV"])].copy()
print("Subset1 size:", len(subset1))
subset1["gt_label"].value_counts()

Subset1 size: 144


gt_label
NV     123
MEL     21
Name: count, dtype: int64

In [22]:
path1 = CSV_OUT / "val_subset1_gt_MEL_or_NV.csv"
subset1.to_csv(path1, index=False)
print("Saved:", path1)

Saved: ../data/isic2018/subsets/val_subset1_gt_MEL_or_NV.csv


## Subset 2: “confusing MEL vs NV”
**choose one of the options (2a, 2b, or 2c) that best fits**

### Option 2a: both probs are non-trivial

In [33]:
tau = 0.10
subset2a = subset1[(subset1["p_MEL"] > tau) & (subset1["p_NV"] > tau)].copy()
print("Subset2a size:", len(subset2a))

Subset2a size: 4


In [29]:
path2 = CSV_OUT / "val_subset2_confusing_a_MEL_NV.csv"
subset2a.to_csv(path2, index=False)
print("Saved:", path2)

Saved: ../data/isic2018/subsets/val_subset2_confusing_a_MEL_NV.csv


### Option 2b: small probability margin

In [30]:
delta = 0.30
subset2b = subset1[(subset1["p_MEL"] - subset1["p_NV"]).abs() < delta].copy()
print("Subset2b size:", len(subset2b))

Subset2b size: 2


In [31]:
path2 = CSV_OUT / "val_subset2_confusing_b_MEL_NV.csv"
subset2b.to_csv(path2, index=False)
print("Saved:", path2)

Saved: ../data/isic2018/subsets/val_subset2_confusing_b_MEL_NV.csv


### Option 2c (cleanest): top-2 are MEL and NV

In [23]:
def top2_labels(row):
    probs = {
        "AK": row["p_AK"], "BCC": row["p_BCC"], "BKL": row["p_BKL"],
        "DF": row["p_DF"], "MEL": row["p_MEL"], "NV": row["p_NV"], "VASC": row["p_VASC"]
    }
    top2 = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:2]
    return [top2[0][0], top2[1][0]]

top2 = subset1.apply(top2_labels, axis=1)
subset1["top1"] = [t[0] for t in top2]
subset1["top2"] = [t[1] for t in top2]

subset2c = subset1[((subset1["top1"]=="MEL") & (subset1["top2"]=="NV")) |
                   ((subset1["top1"]=="NV") & (subset1["top2"]=="MEL"))].copy()
print("Subset2c size:", len(subset2c))

Subset2c size: 59


In [24]:
path2 = CSV_OUT / "val_subset2_confusing_top2_MEL_NV.csv"
subset2c.to_csv(path2, index=False)
print("Saved:", path2)

Saved: ../data/isic2018/subsets/val_subset2_confusing_top2_MEL_NV.csv


## Subset 3: errors MEL↔NV

In [25]:
# If pred_label already exists, use it; otherwise compute from probs
if "pred_label" not in merged.columns:
    labels = ["AK","BCC","BKL","DF","MEL","NV","VASC"]
    merged["pred_label"] = merged[prob_cols].values.argmax(axis=1)
    merged["pred_label"] = merged["pred_label"].map({i: labels[i] for i in range(7)})

subset1 = merged[merged["gt_label"].isin(["MEL","NV"])].copy()

subset3 = subset1[
    ((subset1["gt_label"]=="MEL") & (subset1["pred_label"]=="NV")) |
    ((subset1["gt_label"]=="NV") & (subset1["pred_label"]=="MEL"))
].copy()

print("Subset3 size:", len(subset3))
subset3[["gt_label","pred_label"]].value_counts()

Subset3 size: 8


gt_label  pred_label
MEL       NV            4
NV        MEL           4
Name: count, dtype: int64

In [26]:
path3 = CSV_OUT / "val_subset3_errors_MEL_NV.csv"
subset3.to_csv(path3, index=False)
print("Saved:", path3)

Saved: ../data/isic2018/subsets/val_subset3_errors_MEL_NV.csv


In [35]:
import pytorch_grad_cam
print(dir(pytorch_grad_cam))

['AblationCAM', 'AblationLayer', 'AblationLayerFasterRCNN', 'AblationLayerVit', 'ActivationsAndGradients', 'DeepFeatureFactorization', 'EigenCAM', 'EigenGradCAM', 'FEM', 'FinerCAM', 'FullGrad', 'GradCAM', 'GradCAMElementWise', 'GradCAMPlusPlus', 'GuidedBackpropReLUModel', 'HiResCAM', 'KPCA_CAM', 'LayerCAM', 'RandomCAM', 'ScoreCAM', 'ShapleyCAM', 'XGradCAM', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'ablation_cam', 'ablation_layer', 'activations_and_gradients', 'base_cam', 'eigen_cam', 'eigen_grad_cam', 'feature_factorization', 'fem', 'finer_cam', 'fullgrad_cam', 'grad_cam', 'grad_cam_elementwise', 'grad_cam_plusplus', 'guided_backprop', 'hirescam', 'kpca_cam', 'layer_cam', 'metrics', 'pytorch_grad_cam', 'random_cam', 'run_dff_on_image', 'score_cam', 'shapley_cam', 'utils', 'xgrad_cam']
